# Getting started — Named Entity Recognition

Extract named entities from English text, including your own types, on SageMaker.

**Default types:** PERSON, NORP, FAC, ORG, GPE, LOC, PRODUCT, EVENT, WORK_OF_ART, LAW, LANGUAGE, DATE, TIME, PERCENT, MONEY, QUANTITY, ORDINAL, CARDINAL

This notebook deploys the Sigmodata SageMaker model package from AWS Marketplace, sends example texts, prints spans, then deletes the endpoint.

**Before you run this:**
1. Subscribe on AWS Marketplace (**Continue to Subscribe** → **Continue to Configuration**).
2. Copy the **Model Package ARN** from that page into the next cell. That ARN is AWS's copy of the package (unique per region), not the seller ARN.
3. Run from a SageMaker Notebook / Studio session (execution role auto-detected), or set `role` manually.

Recommended realtime instance: `ml.g4dn.xlarge` (CUDA is used automatically). `ml.m5.large` is the cheap CPU / batch path.

Product page: https://www.sigmodata.com/products/ner-model  
Download this notebook: https://www.sigmodata.com/products/ner-model/getting-started.ipynb

In [ ]:
# Product ARN from AWS Marketplace → Continue to Configuration.
# That is AWS's copy of the model package (unique per region). Do not use a
# seller-account package ARN.
MODEL_PACKAGE_ARN = ""
assert MODEL_PACKAGE_ARN.startswith("arn:aws:sagemaker:"), (
    "Paste the Model Package ARN from Marketplace Continue to Configuration"
)
INSTANCE_TYPE = "ml.g4dn.xlarge"

## 1. Deploy a real-time endpoint

Persistent endpoints bill per hour while running. For bulk files, skip to **Batch transform** — you only pay for job runtime.

In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model import ModelPackage
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

session = sagemaker.Session()
role = get_execution_role()

model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)
predictor = model.deploy(initial_instance_count=1, instance_type=INSTANCE_TYPE)
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

## 2. Extract default entity types

**Request:** `{"input": ["...", "..."]}`  
**Response:** `{"results": [[{label, text, start, end, score}, ...], ...]}` — one inner array per input, in order.

The JSON key is `input`, not `texts` / `inputs`.

In [ ]:
texts = [
    "Apple reported $5 million in revenue yesterday in Cupertino",
    "Barack Obama was born in Honolulu, Hawaii on August 4, 1961.",
]

result = predictor.predict({"input": texts})
for text, ents in zip(texts, result["results"]):
    print(text)
    for e in ents:
        print(f"  {e['label']:12s} {e['text']!r:30s} [{e['start']}:{e['end']}] {e['score']:.2f}")
    print()

## 3. Custom types at inference

Pass `labels` to extract any types without retraining. Optional `threshold` (default 0.5).

In [ ]:
custom = predictor.predict({
    "input": ["Start metformin 500 mg daily for type 2 diabetes."],
    "labels": ["drug", "dosage", "condition"],
    "threshold": 0.5,
})
print(custom)

## 4. boto3 invoke (same contract)

Use this from any Python process that already has an endpoint name.

In [ ]:
import json
import boto3

runtime = boto3.client("sagemaker-runtime", region_name=session.boto_region_name)
resp = runtime.invoke_endpoint(
    EndpointName=predictor.endpoint_name,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps({"input": texts}),
)
print(json.loads(resp["Body"].read()))

## 5. Clean up the endpoint

Real-time endpoints keep billing until deleted. Always run this when finished.

In [ ]:
predictor.delete_endpoint()

## Batch transform (optional)

Cheaper for bulk jobs: CSV in S3 (one text per line, no header) → `index,label,text,start,end,score` CSV out. Use `ml.m5.large`.

In [ ]:
import boto3

bucket = session.default_bucket()
input_key = "sigmodata-example/ner-model/texts.csv"

sample = "\n".join(texts) + "\n"
boto3.client("s3").put_object(Bucket=bucket, Key=input_key, Body=sample.encode())

input_s3_uri = f"s3://{bucket}/{input_key}"
output_s3_uri = f"s3://{bucket}/sigmodata-example/ner-model/output/"
print("Input:", input_s3_uri)
print("Output:", output_s3_uri)

In [ ]:
transformer = model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=output_s3_uri,
    strategy="MultiRecord",
    max_payload=6,
    accept="text/csv",
)

transformer.transform(
    data=input_s3_uri,
    content_type="text/csv",
    split_type="Line",
    wait=True,
)

In [ ]:
output_key = f"{input_key.rsplit('/', 1)[-1]}.out"
obj = boto3.client("s3").get_object(
    Bucket=bucket, Key=f"sigmodata-example/ner-model/output/{output_key}"
)
print(obj["Body"].read().decode())

## Next steps

- Over capacity returns HTTP 429 `too_many_requests` with `Retry-After: 1` instead of dropping the connection. Scale out with more instances.
- Product page with live snippets: https://www.sigmodata.com/products/ner-model
- Support: support@sigmodata.com